# Dolos: one baseline and one final model

Dolos scores bank transfers for human review.
This notebook compares two models on the same test transfers.
The baseline uses transaction fields only.
The final model uses a GraphSAGE encoder and one XGBoost classifier.

**This project is a prototype. It is not financial advice or proof of a crime.**

The default run uses generated data from this project.
It does not use IBM-trained weights.
Do not apply the results to real bank statements.

## Language and technical terms

Use ASD-STE100, Issue 9, for all English text in this notebook.
Use short sentences and the active voice.
Use each technical term with the meaning in this table.
The term list supplies project-specific technical names.

| Technical term | Meaning |
| --- | --- |
| Transfer | One payment from a sender to a receiver. |
| Baseline | The comparison model. It does not use graph history. |
| Graph | Accounts and the transfers that connect the accounts. |
| Node | One account in the graph. |
| Edge | One transfer in the graph. |
| GraphSAGE | The neural encoder that reads past incoming and outgoing transfers. |
| Encoder | The part that converts graph history into numbers. |
| Embedding | The numbers that describe an account's past graph context. |
| XGBoost | The tree classifier that makes a decision from input numbers. |
| Hybrid | The final GraphSAGE-to-XGBoost model. |
| Epoch | One training pass through the selected training batches. |
| Checkpoint | Saved encoder weights from one epoch. |
| Validation data | Data used to select the encoder checkpoint. |
| Test data | Data not used to fit weights or select the checkpoint. |
| Raw margin | The final classifier score before a probability conversion. |
| TreeSHAP | A method that assigns signed feature contributions to a tree score. |
| Average precision | A score that measures the order of positive labels across a ranked list. |
| Precision at 100 | The fraction of positive labels among the first 100 results. |

The model uses a margin above zero as a positive decision.
A positive decision requests review.
It does not establish money laundering.

## Start the notebook

1. Open the repository in a Nix development shell.
2. Install the notebook packages with the command below.
3. Start JupyterLab from that shell.
4. Run the cells in sequence.

```sh
nix develop
uv run --extra ml --with jupyterlab jupyter lab
```

The `ml` extra supplies the training packages.
The website does not need those training packages for inference.

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone
from uuid import uuid4

from IPython.display import Markdown, display
from dolos.experiment import run_experiment

ROOT = Path.cwd()
if not (ROOT / "dolos").is_dir():
    raise RuntimeError("Start this notebook from the repository root.")

## Select the data

The default source contains 2,880 generated transfers.
All account names and transfers in that source are fictional.
The generated data make this notebook easy to run.
They do not supply an independent test of real-world performance.

To use IBM AMLworld HI-Small, set `DATA_KIND` to `ibm`.
Put `HI-Small_Trans.csv` in the repository root.
The full IBM file needs more memory and time.
Do not compare results from different data sources as one experiment.

In [2]:
DATA_KIND = "synthetic"
CSV_PATH = (
    ROOT / "dolos/demo_assets/synthetic-transfers.csv"
    if DATA_KIND == "synthetic"
    else ROOT / "HI-Small_Trans.csv"
)
EPOCHS = 8
BATCH_SIZE = 192
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIRECTORY = ROOT / "var/notebook-runs" / f"{DATA_KIND}-{run_stamp}-{uuid4().hex[:6]}"
if not CSV_PATH.is_file():
    raise FileNotFoundError("The selected CSV file is not available.")
display(Markdown(f"**Data source:** `{DATA_KIND}`. **Encoder training limit:** {EPOCHS} epochs."))

**Data source:** `synthetic`. **Encoder training limit:** 8 epochs.

## Define the comparison

The baseline uses amounts, currencies, and payment format.
It uses one XGBoost classifier.
It does not use the account graph.

The final model has two stages.
GraphSAGE makes 64 embedding values for each endpoint.
One XGBoost classifier reads those 128 values and the transaction fields.
That classifier makes the final decision.
The two stages do not vote as separate models.

```text
Baseline: transaction fields -> XGBoost -> score

Final model:
  past graph -> GraphSAGE -> endpoint embeddings --+
  transaction fields -----------------------------+-> XGBoost -> score
```

The encoder uses two layers.
Each layer has separate incoming and outgoing means.
The first stage samples four neighbors in each direction.
The next stage samples two neighbors in each direction.

## Prevent future data use

The runner sorts transfers by time and source row ID.
It places the data boundaries at the start of an hour.
The first 70 percent supply training data.
The next 15 percent supply validation data.
The last 15 percent supply test data.
Hour boundaries can change the exact group sizes.

The encoder uses only transfers before the target hour.
Both graph layers use the same time limit.
The runner fits category codes on training data only.
Unknown categories use a separate code.

The runner uses validation average precision to select the encoder checkpoint.
It then keeps that encoder fixed.
It fits the final XGBoost classifier on training transfers only.
It does not use test labels to fit weights or select a checkpoint.

## Train the two models

Run the next cell to fit the baseline and the final model.
The runner creates a new output directory.
It does not replace the website's deployed weights.
The full encoder log stays in that directory.
The comparison below contains only two result rows.

In [3]:
report, predictions = run_experiment(
    CSV_PATH,
    RUN_DIRECTORY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    data_kind=DATA_KIND,
)
assert set(report["models"]) == {"transaction_xgboost", "hybrid"}
assert report["models"]["transaction_xgboost"]["rows"] == report["models"]["hybrid"]["rows"]
display(Markdown(f"**Data type:** `{report['data_kind']}`. **Source rows:** {report['source_rows']:,}."))

**Data type:** `synthetic`. **Source rows:** 2,880.

In [4]:
split_lines = ["| Data group | Rows | Positive labels |", "| --- | ---: | ---: |"]
for name, values in report["splits"].items():
    split_lines.append(f"| {name.title()} | {values['rows']:,} | {values['positives']:,} |")
display(Markdown("\n".join(split_lines)))
display(Markdown(f"**Selected encoder checkpoint:** epoch {report['selected_encoder_epoch']}."))

| Data group | Rows | Positive labels |
| --- | ---: | ---: |
| Train | 2,016 | 672 |
| Validation | 432 | 144 |
| Test | 432 | 144 |

**Selected encoder checkpoint:** epoch 1.

## Compare the test results

Use the same test transfers and review limit for both models.
Higher average precision means better ranking on this test.
Precision at 100 measures only the first 100 results.
It is not accuracy across all transfers.

The numbers below come from this notebook run.
They are not the saved results from the earlier IBM benchmark.

In [5]:
names = {"transaction_xgboost": "Baseline: transaction-only XGBoost", "hybrid": "Final: GraphSAGE to XGBoost"}
result_lines = [
    "| Model | Average precision | Precision at 100 | Test rows | Positive decisions |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for name, result in report["models"].items():
    precision = result.get("precision_at_100")
    precision_text = f"{precision:.3f}" if precision is not None else "Not available"
    result_lines.append(
        f"| {names[name]} | {result['average_precision']:.6f} | {precision_text} | "
        f"{result['rows']:,} | {result['positive_decisions']:,} |"
    )
display(Markdown("\n".join(result_lines)))
display(Markdown(report["evaluation_note"]))

| Model | Average precision | Precision at 100 | Test rows | Positive decisions |
| --- | ---: | ---: | ---: | ---: |
| Baseline: transaction-only XGBoost | 0.976999 | 0.980 | 432 | 151 |
| Final: GraphSAGE to XGBoost | 1.000000 | 1.000 | 432 | 144 |

The data use the same generator as training. This test does not measure real-world performance.

## Inspect one final decision

Select the highest score from the final model.
The score has two feature groups.
One group contains GraphSAGE embedding values.
The other group contains direct transaction fields.

XGBoost supplies all TreeSHAP contributions in this decision.
The two group contributions and the base margin add to the final margin.
The contributions describe the classifier's use of its inputs.
They do not assign causal responsibility to a person or a transfer.

In [6]:
sample = predictions["hybrid"]["scores"][0]
credits = sample["component_contributions"]
credit_sum = sample["base_margin"] + sum(item["contribution"] for item in credits)
assert abs(credit_sum - sample["score"]) < 0.0001
credit_lines = ["| Score part | Raw margin contribution |", "| --- | ---: |"]
credit_lines.append(f"| Base margin | {sample['base_margin']:+.6f} |")
for item in credits:
    credit_lines.append(f"| {item['name']} | {item['contribution']:+.6f} |")
credit_lines.append(f"| Final margin | {sample['score']:+.6f} |")
display(Markdown("\n".join(credit_lines)))

| Score part | Raw margin contribution |
| --- | ---: |
| Base margin | -0.810358 |
| GraphSAGE neighborhood | +7.816066 |
| Transaction fields | +0.656824 |
| Final margin | +7.662531 |

## Check the graph context

The next cell checks the actual sampled historical transfers.
Each context time must be before the target hour.
The context shows which transfers the encoder could read.
Context membership does not measure a transfer's causal effect.
The first three tree paths show actual classifier branches.

In [7]:
detail = sample["explanation"]
assert detail["kind"] == "hybrid"
for row in detail["context_transactions"]:
    context_time = datetime.fromisoformat(row["timestamp"]).timestamp() * 1_000_000
    assert context_time < sample["context_cutoff_us"]
display(Markdown(
    f"**Sampled context transfers:** {len(detail['context_transactions'])}. "
    f"**Tree paths supplied:** {len(detail['tree_paths'])}."
))
assert detail["tree_paths"]

**Sampled context transfers:** 12. **Tree paths supplied:** 3.

## Reuse the trained files

The output directory contains the baseline, encoder, and final classifier files.
It also contains source checksums, split sizes, model weights, and test scores.
The final model needs these four files:

- `manifest.json`
- `graphsage.npz`
- `hybrid_manifest.json`
- `hybrid_xgboost.json`

Keep those files together.
The API checks the encoder and classifier checksums.
Do not combine files from different training runs.
Set `DOLOS_MODEL_DIR` to this directory to test uploaded canonical transactions.
A website example also needs a matching source file and graph description.

In [8]:
required = ("manifest.json", "graphsage.npz", "hybrid_manifest.json", "hybrid_xgboost.json")
assert all((RUN_DIRECTORY / name).is_file() for name in required)
relative_directory = RUN_DIRECTORY.relative_to(ROOT)
display(Markdown(f"**Output directory:** `{relative_directory}`."))
display(Markdown(f"**Source checksum:** `{report['source_sha256']}`."))

**Output directory:** `var/notebook-runs/synthetic-20261116T011308Z-863962`.

**Source checksum:** `314fb67b8e4ce8e16c3873f61e1978ee9a5383945382cd2fd9ff81d276ceedfb`.

## Limits and next work

The default data contain simple generated patterns.
A high score on this source does not establish real-world performance.
Compare the baseline and final model on the same independent data next.
Measure false alerts and missed positive labels.
Test new banks, currencies, and statement formats.
Do not use a model flag as proof of a crime.

The project keeps earlier model code for tests and notebook history.
This notebook reports only the baseline and the final hybrid.

## Sources and tools

- [IBM AMLworld](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml) supplies the optional IBM source.
- [The project generator](dolos/model_demo.py) supplies the default synthetic source.
- [The experiment runner](dolos/experiment.py) fits and compares the two models.
- [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) supplies GraphSAGE layers for encoder training.
- [XGBoost](https://xgboost.readthedocs.io/) supplies the baseline and final tree classifier.
- [ASD-STE100](https://www.asd-ste100.org/assets/files/ASD-STE100_ISSUE9.pdf) defines the language standard.
- OpenAI Codex and ChatGPT assisted code changes, tests, and text.

The final hybrid is a new combination of notebook components.
It is not the earlier IBM-trained baseline model.